# Sequence Modeling

## Begin by deciding what the sequence task predicts

A sequence model is defined not only by its architecture but by the relationship between input positions and target positions. Predicting the next sensor reading, labeling every token, and classifying an entire event history are different tasks, even when they use the same recurrent layer.

This lesson organizes common sequence tasks, makes input/target alignment explicit, and explains the split and masking decisions that keep evaluation honest.

## Match the readout to the task

| Task | Input | Prediction | Example |
| --- | --- | --- | --- |
| Sequence classification | Whole sequence | One label | Is this machine run faulty? |
| Sequence labeling | Whole sequence | One label per position | Tag each word or event |
| Forecasting | Past observations | Future value(s) | Predict tomorrow's demand |
| Sequence generation | Prefix so far | Next symbol/value, repeated | Generate a continuation |

A many-to-one model summarizes a sequence before a single output. A many-to-many model emits outputs across positions. The output head and loss must reflect the target shape; a classifier should not silently treat padding as a real final observation.

## Make the time alignment visible

For one-step forecasting, a window of length $k$ predicts the next point:

$$[x_{t-k+1},\ldots,x_t]\longrightarrow x_{t+1}.$$

For a two-step horizon, the same input predicts $[x_{t+1},x_{t+2}]$. For sequence labeling, input and target positions align instead: $x_t\to y_t$, subject to any task-specific delay. Print a few windows and targets before training; off-by-one errors can produce plausible but meaningless scores.

## A tiny forecast window

Suppose the observations are $[2,3,5,4,6]$. With a window length of three, the first pair is input $[2,3,5]$ and target $4$; the next is $[3,5,4]$ and target $6$. The window is a supervised training example, not an independent time series. Adjacent windows overlap and therefore should not be randomly split across train and test.

For a chronological evaluation, choose target-time boundaries first. A validation window may use earlier observations as context, but its target must be after the training boundary. Fit scaling statistics on training observations only, and compare against a persistence baseline such as predicting the last observed value.

## Generation and teacher forcing

An autoregressive model estimates a next-step distribution conditioned on its prefix, $p(x_t\mid x_{<t})$. During teacher-forced training, the true previous value is supplied as context. During generation, the model consumes its own previous prediction or sampled token, so errors can accumulate. Evaluation must specify whether it measures one-step predictions with true context or free-running multi-step generation.

For classification and labeling, use appropriate masks for padded elements and exclude ignored target positions from the loss. For forecasting, define the horizon and report suitable metrics in original units as well as any normalized loss.

**Takeaway:** define target timing and the evaluation protocol before choosing an RNN, convolution, or Transformer. Split by time or source, keep preprocessing inside the training data, and compare with a meaningful baseline.